## Q1: Roles of Driver, Cluster Manager, and Executor
Driver

The Driver is the main program that coordinates the Spark application. It:

Creates the SparkSession/SparkContext.
Converts transformations into a DAG.
Schedules tasks.
Communicates with executors.
Cluster Manager

The Cluster Manager manages cluster resources. It allocates CPU and memory to the Spark application.

Examples include:

Standalone
YARN
Kubernetes
Executor

An Executor runs tasks on worker nodes and stores data in memory or on disk for caching.

Driver
   ↓
Cluster Manager
   ↓
Executors on Worker Nodes

## Q2: How does Lazy Evaluation improve performance?

Spark does not immediately execute transformations such as:
df.filter(...)
df.select(...)
df.groupBy(...)

Instead, Spark builds a DAG (Directed Acyclic Graph) of operations.

Execution begins only when an action such as show(), count(), or write() is called.

This allows Spark to:

Combine multiple operations.
Optimize the execution plan.
Avoid unnecessary computations.
Reduce data movement and disk I/O.

result = (
    df.filter(df["price"] > 100)
      .select("product_id", "price")
)

result.show()

The operations are optimized and executed together only when show() is called.

## Q3: Reading a CSV File

In [ ]:
df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

## Q4: CSV vs Parquet

CSV is a row-based text format, while Parquet is a columnar binary storage format.

CSV files are easy to read and share but usually require more storage and processing time. Parquet stores data by columns, allowing Spark to read only the columns required by a query.

For example, if a query needs only product_id and price, Parquet can read only those columns. This reduces disk I/O, network transfer, and memory usage, resulting in better performance for analytical workloads.

## Q5: Selecting Electronics Products

In [ ]:
from pyspark.sql.functions import col

result = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)

result.show()

## Q6: Renaming and Casting Columns

In [ ]:
from pyspark.sql.functions import col

df_revised = (
    df
    .withColumnRenamed("old_name", "new_name")
    .withColumn(
        "price",
        col("price").cast("double")
    )
)

## Q7: DAG and Fault Tolerance

Spark maintains a Lineage Graph, also called a DAG, that records the sequence of transformations used to create data.

If a worker node fails and a partition is lost, Spark uses the lineage information to recompute only the lost partition from the original data and transformations.

Therefore, Spark does not need to restart the entire application, providing fault tolerance.

## Q8: Filtering Completed Orders

In [ ]:
from pyspark.sql.functions import col

result = df_orders.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

result.show()

## Q9: Predicate Pushdown

Predicate Pushdown is an optimization technique in which Spark pushes filter conditions closer to the data source.

For example, when reading a Parquet file and filtering rows where amount is greater than 1000, Spark can apply the filter while reading the data instead of loading the entire dataset first.

This reduces the amount of data read from storage and loaded into memory. As a result, disk I/O, network traffic, memory usage, and processing time are reduced.

## Q10: Calculating Final Price

In [ ]:
from pyspark.sql.functions import col

df_final = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

## Q11: Transformations and Actions

Transformations create a new DataFrame or RDD and are lazily evaluated. They do not immediately execute the computation.

Examples of transformations:
1. filter()
2. select()

Other examples include groupBy(), join(), and withColumn().

Actions trigger the actual execution of Spark computations.

Examples of actions:
1. show()
2. count()

Other examples include collect(), first(), and write().

## Q12: Parquet to CSV Processing Pipeline

In [ ]:
from pyspark.sql.functions import col

result = (
    spark.read
    .parquet("path/to/input")
    .filter(
        col("user_id").isNotNull()
    )
)

result.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("path/to/output")

## Q13: Client Mode vs Cluster Mode

In Client Mode, the Driver runs on the machine from which the Spark application is submitted. This mode is useful for interactive development and debugging.

In Cluster Mode, the Driver runs inside the cluster. This mode is more suitable for production applications and long-running jobs.

The main difference is the location of the Driver program.

## Q14: Filtering by Region or Priority

In [ ]:
from pyspark.sql.functions import col

result = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

result.show()

## Q15: show(5) vs collect()

show(5) displays only five rows from the DataFrame and is safer when exploring a very large dataset.

collect() retrieves all rows and brings them to the Driver node. For a multi-terabyte dataset, this can consume all available Driver memory and cause an OutOfMemoryError.

Therefore, show(5) is safer because it limits the amount of data returned and avoids loading the entire dataset into the Driver's memory.

Week6_Spark_Questions.ipynb
│
├── Q1 - Theory
├── Q2 - Theory
├── Q3 - CSV Read Code
├── Q4 - CSV vs Parquet
├── Q5 - Filter and Select Code
├── Q6 - Rename and Cast Code
├── Q7 - DAG and Fault Tolerance
├── Q8 - Filter Orders Code
├── Q9 - Predicate Pushdown
├── Q10 - Add Column Code
├── Q11 - Transformations vs Actions
├── Q12 - Parquet to CSV Code
├── Q13 - Client vs Cluster Mode
├── Q14 - Filter with OR Code
└── Q15 - show(5) vs collect()

## Final Insights

### Architecture
Spark uses a Driver to coordinate the application, a Cluster Manager to allocate resources, and Executors to perform tasks on worker nodes.

### Performance
Spark improves performance through Lazy Evaluation, DAG optimization, in-memory processing, Predicate Pushdown, and efficient columnar formats such as Parquet.

### Data Processing
The assignment demonstrates a complete data processing workflow involving reading data, transforming DataFrames, filtering records, handling null values, and writing processed results.

### Best Practices
For large datasets, `show()` or `limit()` should be preferred for exploration instead of `collect()`, which can overload the Driver's memory.